In [1]:
import os
import glob
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
import torch
print(torch.__version__)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")


FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
2.3.1+cpu
False no cuda


# 1) إعداد المسارات

In [3]:
DATASET_DIR = r"C:\Users\hp\mona(smartpark)\pklot-parking"
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

In [4]:
assert os.path.exists(DATA_YAML), f"data.yaml مش موجود هنا: {DATA_YAML}"

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

In [5]:
names = data_cfg.get("names", None)
if isinstance(names, dict):
    class_names = [names[i] for i in range(len(names))]
elif isinstance(names, list):
    class_names = names
else:
    raise ValueError("data.yaml لازم يحتوي names (list أو dict)")


In [6]:
num_classes = len(class_names)
bg_name = "background"
all_names = class_names + [bg_name]
BG = num_classes

print("Classes:", class_names)
print("Train:", data_cfg.get("train"))
print("Val  :", data_cfg.get("val"))
print("Test :", data_cfg.get("test"))

Classes: ['space-empty', 'space-occupied']
Train: train/images
Val  : valid/images
Test : test/images


# 2) تدريب YOLOv8 

In [8]:
from ultralytics import YOLO
model = YOLO("yolov8m.pt")

In [ ]:
train_results = model.train(
    data=DATA_YAML,
    imgsz=960,
    epochs=80,
    patience=20,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,
    cache=True,
    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.10,
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
    translate=0.10,
    scale=0.50,
    project="runs_pklot",
    name="yolov8m_pklot_strong",
    verbose=True
)


Ultralytics 8.3.63  Python-3.12.7 torch-2.3.1+cpu CPU (11th Gen Intel Core(TM) i7-1195G7 2.90GHz)
engine\trainer: task=detect, mode=train, model=yolov8m.pt, data=C:\Users\hp\mona(smartpark)\pklot-parking\data.yaml, epochs=80, time=None, patience=20, batch=16, imgsz=960, save=True, save_period=-1, cache=True, device=None, workers=8, project=runs_pklot, name=yolov8m_pklot_strong3, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, 

train: Scanning C:\Users\hp\mona(smartpark)\pklot-parking\train\labels.cache... 8690 images, 189 backgrounds, 0 corrupt


train: 33.6GB RAM required to cache images with 50% safety margin but only 8.7/31.8GB available, not caching images 
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


C:\Users\hp\anaconda3\Lib\site-packages\albumentations\check_version.py:107: UserWarning: Error fetching version info <urlopen error [Errno 11001] getaddrinfo failed>
  data = fetch_version_info()
C:\Users\hp\anaconda3\Lib\site-packages\ultralytics\data\augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning C:\Users\hp\mona(smartpark)\pklot-parking\valid\labels.cache... 2483 images, 59 backgrounds, 0 corrupt: 1


val: 9.6GB RAM required to cache images with 50% safety margin but only 8.7/31.8GB available, not caching images 
Plotting labels to runs_pklot\yolov8m_pklot_strong3\labels.jpg... 
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 960 train, 960 val
Using 0 dataloader workers
Logging results to runs_pklot\yolov8m_pklot_strong3
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/80         0G      2.676      3.479      2.221       1433        960:   3%|▎         | 14/544 [14:33<8:31:21, 

In [ ]:
save_dir = str(model.trainer.save_dir)
best_pt = os.path.join(save_dir, "weights", "best.pt")
print("\nBest model saved at:", best_pt)


In [ ]:
best_model = YOLO(best_pt)